# AIC 2026 — Speech-to-Text với VinAI PhoWhisper Large

Notebook quét video, chạy model VinAI PhoWhisper Large, rồi lưu JSON, SRT và TXT. File đã hoàn thành sẽ được bỏ qua để có thể tiếp tục khi runtime ngắt kết nối.

Notebook chạy được trên **cả Google Colab và Kaggle**, tự phát hiện môi trường:

| | Colab | Kaggle |
|---|---|---|
| Video | `MyDrive/AI Challenge/Dataset_Directory` | `/kaggle/input/datasets/fatle542/aic-dataset` |
| Transcript | `MyDrive/AI Challenge/Transcripts` | `/kaggle/working/Transcripts` |

**Colab**: Runtime → Change runtime type → GPU (A100/L4 khuyến nghị).

**Kaggle**: Settings → Accelerator → **GPU T4 x2 / P100**, và bật **Internet: On** (cần tải model từ Hugging Face). Lưu ý `/kaggle/input` là read-only nên kết quả ghi vào `/kaggle/working`; nhớ tải về hoặc *Save Version* trước khi hết session, vì `/kaggle/working` bị xoá khi session kết thúc.

In [ ]:
import os
import shutil
import subprocess
import sys

def detect_env():
    """Phát hiện môi trường: 'kaggle' | 'colab' | 'local'.

    Kiểm tra Kaggle TRƯỚC vì /kaggle là dấu hiệu chắc chắn; image của Kaggle có
    thể khiến các tín hiệu của Colab khớp sai.
    """
    if os.path.isdir('/kaggle/input') or os.path.isdir('/kaggle/working') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    if os.environ.get('COLAB_RELEASE_TAG') or os.path.isdir('/content') or 'google.colab' in sys.modules:
        return 'colab'
    return 'local'

ENV = detect_env()
print('Môi trường:', ENV)
print('  /kaggle/input:', os.path.isdir('/kaggle/input'),
      '| KAGGLE_KERNEL_RUN_TYPE:', os.environ.get('KAGGLE_KERNEL_RUN_TYPE'),
      '| /content:', os.path.isdir('/content'),
      '| COLAB_RELEASE_TAG:', os.environ.get('COLAB_RELEASE_TAG'))

subprocess.run(['nvidia-smi'], check=False)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-U', 'transformers', 'accelerate'], check=True)

# Kaggle đã có ffmpeg sẵn; chỉ cài khi thiếu.
if shutil.which('ffmpeg') is None:
    subprocess.run('apt-get -qq update && apt-get -qq install -y ffmpeg', shell=True, check=True)
print('ffmpeg:', shutil.which('ffmpeg'))

In [ ]:
# Chỉ Colab cần mount Drive. Trên Kaggle dataset đã có sẵn ở /kaggle/input.
if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Bỏ qua mount Drive (ENV =', ENV, ')')

## Cấu hình

Đường dẫn được chọn tự động theo môi trường:

- **Colab**: đọc `AI Challenge/Dataset_Directory`, ghi vào `AI Challenge/Transcripts` trên Drive.
- **Kaggle**: đọc `/kaggle/input/datasets/fatle542/aic-dataset` (read-only), ghi vào `/kaggle/working/Transcripts`.

`TARGET_FOLDERS` là danh sách thư mục video cần xử lý. Để `TARGET_FOLDERS = ['.']` nếu muốn quét toàn bộ dataset. Cell sẽ in ra các thư mục thực có để bạn đối chiếu (cấu trúc dataset trên Kaggle có thể lồng thêm một cấp).

In [ ]:
import os
import sys
from pathlib import Path

# ENV được đặt ở cell đầu; tính lại nếu kernel vừa restart.
# Muốn ép môi trường thì gán thẳng ở đây, ví dụ: ENV = 'kaggle'
try:
    ENV
except NameError:
    if os.path.isdir('/kaggle/input') or os.path.isdir('/kaggle/working') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        ENV = 'kaggle'
    elif os.environ.get('COLAB_RELEASE_TAG') or os.path.isdir('/content'):
        ENV = 'colab'
    else:
        ENV = 'local'
print('Môi trường:', ENV)

#@title Đường dẫn theo môi trường
if ENV == 'kaggle':
    # Kaggle: /kaggle/input là read-only nên transcript phải ghi ra /kaggle/working.
    KAGGLE_DATASET_CANDIDATES = [
        Path('/kaggle/input/datasets/fatle542/aic-dataset'),
        Path('/kaggle/input/aic-dataset'),
    ]
    DATASET_DIRECTORY = next((p for p in KAGGLE_DATASET_CANDIDATES if p.is_dir()), KAGGLE_DATASET_CANDIDATES[0])
    TRANSCRIPTS_DIRECTORY = Path('/kaggle/working/Transcripts')
elif ENV == 'colab':
    DATASET_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Dataset_Directory')
    TRANSCRIPTS_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Transcripts')
else:
    DATASET_DIRECTORY = Path('./Dataset_Directory')
    TRANSCRIPTS_DIRECTORY = Path('./Transcripts')

assert DATASET_DIRECTORY.is_dir(), (
    f'Không tìm thấy dataset: {DATASET_DIRECTORY}\n'
    + ('Kiểm tra tên dataset đã Add vào notebook. Có trong /kaggle/input: '
       + ', '.join(sorted(p.name for p in Path('/kaggle/input').iterdir())) if ENV == 'kaggle' else '')
)

# Chỉ giữ lại các thư mục bạn muốn chuyển giọng nói thành văn bản trong mảng này.
# Dùng ['.'] để quét toàn bộ dataset.
TARGET_FOLDERS = [
    'Videos_L29_a',
    'Videos_L30_a',
]

MODEL_ID = 'vinai/PhoWhisper-large'
OVERWRITE = False
VIDEO_EXTENSIONS = {'.mp4', '.mkv', '.mov', '.avi', '.webm', '.m4v', '.mpeg', '.mpg'}

dataset_root_resolved = DATASET_DIRECTORY.resolve()

# Liệt kê các thư mục con thực có (2 cấp) để đối chiếu — cấu trúc trên Kaggle có thể lồng thêm một cấp.
AVAILABLE_VIDEO_FOLDERS = sorted(
    str(p.relative_to(dataset_root_resolved))
    for depth in ('*', '*/*')
    for p in dataset_root_resolved.glob(depth)
    if p.is_dir()
)
print('Dataset:', dataset_root_resolved)
print('Thư mục con hiện có:')
for folder in AVAILABLE_VIDEO_FOLDERS[:40]:
    print('  -', folder)
if len(AVAILABLE_VIDEO_FOLDERS) > 40:
    print(f'  ... và {len(AVAILABLE_VIDEO_FOLDERS) - 40} thư mục khác')

def resolve_target(folder):
    """Trả về đường dẫn tuyệt đối của thư mục video, tìm cả ở cấp lồng bên trong."""
    relative_folder = Path(str(folder).strip() or '.')
    assert not relative_folder.is_absolute(), f'Thư mục phải là đường dẫn tương đối: {folder}'
    candidate = (DATASET_DIRECTORY / relative_folder).resolve()
    if not candidate.is_dir():
        # Dataset trên Kaggle thường bọc thêm một thư mục gốc → tìm theo tên.
        matches = [p for p in dataset_root_resolved.glob(f'*/{relative_folder}') if p.is_dir()]
        assert matches, (
            f'Không tìm thấy thư mục: {folder}\n'
            f'Các thư mục hiện có: {AVAILABLE_VIDEO_FOLDERS[:40]}'
        )
        candidate = matches[0].resolve()
    assert candidate == dataset_root_resolved or dataset_root_resolved in candidate.parents, \
        f'Thư mục không được nằm ngoài dataset: {folder}'
    return candidate

assert TARGET_FOLDERS, 'TARGET_FOLDERS không được để trống'
VIDEO_ROOTS = [resolve_target(folder) for folder in TARGET_FOLDERS]

VIDEO_ROOT = dataset_root_resolved  # Thư mục gốc chung để giữ cấu trúc khi lưu kết quả
TRANSCRIPTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = TRANSCRIPTS_DIRECTORY.resolve()

print('\nCác thư mục đầu vào:')
for root in VIDEO_ROOTS:
    print('-', root)
print('Output:', OUTPUT_ROOT)

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

assert torch.cuda.is_available(), (
    'Chưa bật GPU: Kaggle → Settings → Accelerator → GPU T4 x2 / P100'
    if ENV == 'kaggle' else
    'Chưa bật GPU: Runtime → Change runtime type → GPU'
)

# Mỗi GPU giữ một bản model riêng và xử lý một video tại một thời điểm.
# Nếu chỉ có một GPU, notebook tự động quay về chế độ single-GPU.
GPU_COUNT = min(2, torch.cuda.device_count())
model_dtype = torch.float16
print(f'Phát hiện {torch.cuda.device_count()} GPU; sẽ dùng {GPU_COUNT} GPU:')
for gpu_id in range(GPU_COUNT):
    print(f'  cuda:{gpu_id}: {torch.cuda.get_device_name(gpu_id)}')

def build_transcriber(gpu_id):
    device = f'cuda:{gpu_id}'
    print(f'[GPU {gpu_id}] Đang tải {MODEL_ID}...')
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID, dtype=model_dtype, low_cpu_mem_usage=True,
    ).to(device)
    asr = pipeline(
        'automatic-speech-recognition',
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        dtype=model_dtype,
        device=device,
    )
    print(f'[GPU {gpu_id}] Đã tải model')
    return asr

transcribers = [build_transcriber(gpu_id) for gpu_id in range(GPU_COUNT)]
print(f'Sẵn sàng chạy với {GPU_COUNT} GPU')

In [ ]:
videos = sorted(
    {
        p
        for root in VIDEO_ROOTS
        for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    },
    key=lambda p: str(p).lower(),
)
print(f'Tìm thấy {len(videos)} video')
for video in videos[:10]:
    print('-', video.relative_to(VIDEO_ROOT))

In [ ]:
import json
import subprocess
import traceback

import numpy as np

SAMPLE_RATE = 16_000

def load_audio(path, sampling_rate=SAMPLE_RATE):
    """Giải mã audio thành mảng float32 mono bằng ffmpeg.

    Không truyền đường dẫn trực tiếp cho pipeline: transformers sẽ đọc cả file
    thành bytes rồi bơm qua stdin của ffmpeg, mà MP4 cần input seekable nên trên
    một số bản ffmpeg (Kaggle) sẽ trả về 0 byte. Đọc thẳng từ file thì luôn ổn.
    """
    command = [
        'ffmpeg', '-nostdin', '-threads', '0',
        '-i', str(path),
        '-vn', '-f', 'f32le', '-acodec', 'pcm_f32le',
        '-ac', '1', '-ar', str(sampling_rate),
        '-loglevel', 'error', '-',
    ]
    process = subprocess.run(command, capture_output=True)
    if process.returncode != 0:
        stderr = process.stderr.decode('utf-8', 'ignore').strip()
        raise RuntimeError(f'ffmpeg lỗi khi đọc {path}:\n{stderr[-2000:]}')
    audio = np.frombuffer(process.stdout, dtype=np.float32)
    if audio.size == 0:
        raise RuntimeError(f'Không giải mã được audio (video không có tiếng?): {path}')
    return {'raw': audio.copy(), 'sampling_rate': sampling_rate}

def transcribe(asr, video):
    return asr(
        load_audio(video),
        return_timestamps=True,
        generate_kwargs={'language': 'vi', 'task': 'transcribe'},
    )

def srt_timestamp(seconds):
    milliseconds = max(0, round(float(seconds) * 1000))
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, millis = divmod(remainder, 1000)
    return f'{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}'

def format_timestamp(seconds):
    if seconds is None:
        return '??:??:??.???'
    return srt_timestamp(seconds).replace(',', '.')

def save_transcript(video, result):
    relative = video.relative_to(VIDEO_ROOT).with_suffix('')
    stem = OUTPUT_ROOT / relative
    stem.parent.mkdir(parents=True, exist_ok=True)
    segments = []
    for index, chunk in enumerate(result.get('chunks', [])):
        timestamp = chunk.get('timestamp') or (0.0, 0.0)
        start = float(timestamp[0] or 0.0)
        end = float(timestamp[1] if timestamp[1] is not None else start)
        segments.append({
            'id': index,
            'start': start,
            'end': end,
            'video_start': start,
            'video_end': end,
            'text': chunk.get('text', '').strip(),
        })
    payload = {
        'source': str(video),
        'video_id': video.stem,
        'model': MODEL_ID,
        'language': 'vi',
        'text': result.get('text', '').strip(),
        'segments': segments,
    }
    stem.with_suffix('.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    timestamped_text = '\n'.join(
        f"[{format_timestamp(segment['video_start'])} --> {format_timestamp(segment['video_end'])}] {segment['text']}"
        for segment in segments
    )
    stem.with_suffix('.txt').write_text(
        (timestamped_text or payload['text']) + '\n', encoding='utf-8'
    )
    blocks = []
    for index, segment in enumerate(payload['segments'], 1):
        text = segment.get('text', '').strip()
        blocks.append(f"{index}\n{srt_timestamp(segment['start'])} --> {srt_timestamp(segment['end'])}\n{text}")
    stem.with_suffix('.srt').write_text('\n\n'.join(blocks) + ('\n' if blocks else ''), encoding='utf-8')
    return stem

def output_json_path(video):
    return (OUTPUT_ROOT / video.relative_to(VIDEO_ROOT)).with_suffix('.json')

## Chạy thử một video

Nên chạy cell này trước để xác nhận đường dẫn, GPU và chất lượng transcript.

In [ ]:
assert videos, 'Không tìm thấy video nào'
sample_video = videos[0]

# Kiểm tra khâu giải mã audio trước khi chạy model — lỗi ffmpeg sẽ lộ ra ngay ở đây.
sample_audio = load_audio(sample_video)
print('Video:', sample_video)
print(f"Audio: {sample_audio['raw'].size / SAMPLE_RATE:.1f}s @ {sample_audio['sampling_rate']} Hz")

sample_result = transcribe(transcribers[0], sample_video)
sample_stem = save_transcript(sample_video, sample_result)
print('Đã lưu:', sample_stem)
print('\n--- KẾT QUẢ CÓ TIMESTAMP TRONG VIDEO ---\n')
for chunk in sample_result.get('chunks', []):
    start, end = chunk.get('timestamp') or (None, None)
    print(f"[{format_timestamp(start)} --> {format_timestamp(end)}] {chunk.get('text', '').strip()}")
if not sample_result.get('chunks'):
    print(sample_result.get('text', '')[:1000])

## Chạy toàn bộ video

Cell này tự bỏ qua các video đã có `.json`. Nếu runtime ngắt kết nối, chỉ cần chạy lại các cell setup (mount Drive nếu ở Colab, load model) rồi chạy lại cell này.

Trên Kaggle, session có giới hạn ~9–12 giờ và `/kaggle/working` bị xoá khi session kết thúc — chạy cell đóng gói ở cuối để tải kết quả về, hoặc *Save Version* để giữ output.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from queue import Queue, Empty
from threading import Lock

jobs = Queue()
for index, video in enumerate(videos, 1):
    jobs.put((index, video))

print_lock = Lock()

def output_is_complete(video):
    json_path = output_json_path(video)
    stem = json_path.with_suffix('')
    if not all(stem.with_suffix(ext).exists() for ext in ('.json', '.txt', '.srt')):
        return False, None
    try:
        existing_model = json.loads(json_path.read_text(encoding='utf-8')).get('model')
    except Exception:
        return False, None
    return existing_model == MODEL_ID, existing_model

def gpu_worker(gpu_id):
    asr = transcribers[gpu_id]
    local_success = local_skipped = local_failed = 0
    local_failures = []
    with torch.cuda.device(gpu_id):
        while True:
            try:
                index, video = jobs.get_nowait()
            except Empty:
                break
            try:
                complete, existing_model = output_is_complete(video)
                if complete and not OVERWRITE:
                    local_skipped += 1
                    with print_lock:
                        print(f'[GPU {gpu_id}] [{index}/{len(videos)}] SKIP {video.name}')
                    continue
                if existing_model is not None and not OVERWRITE:
                    with print_lock:
                        print(f'[GPU {gpu_id}] [{index}/{len(videos)}] REPROCESS {video.name} (model cũ: {existing_model})')
                with print_lock:
                    print(f'[GPU {gpu_id}] [{index}/{len(videos)}] STT  {video}')
                result = transcribe(asr, video)
                save_transcript(video, result)
                local_success += 1
            except Exception as error:
                local_failed += 1
                local_failures.append({'video': str(video), 'gpu': gpu_id, 'error': repr(error)})
                with print_lock:
                    print(f'[GPU {gpu_id}] ERROR {video}: {error!r}')
                    print(traceback.format_exc())
            finally:
                jobs.task_done()
                torch.cuda.empty_cache()
    return local_success, local_skipped, local_failed, local_failures

with ThreadPoolExecutor(max_workers=GPU_COUNT) as executor:
    worker_results = list(executor.map(gpu_worker, range(GPU_COUNT)))

success = sum(item[0] for item in worker_results)
skipped = sum(item[1] for item in worker_results)
failed = sum(item[2] for item in worker_results)
failed_videos = [failure for item in worker_results for failure in item[3]]
(OUTPUT_ROOT / '_failed.json').write_text(
    json.dumps(failed_videos, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'Hoàn tất với {GPU_COUNT} GPU: success={success}, skipped={skipped}, failed={failed}, total={len(videos)}')

## Đóng gói kết quả (Kaggle)

Nén toàn bộ transcript thành một file zip trong `/kaggle/working` để tải về từ tab **Output**. Trên Colab kết quả đã nằm sẵn trên Drive nên cell này chỉ báo bỏ qua.

In [ ]:
if ENV == 'kaggle':
    import shutil

    archive = shutil.make_archive('/kaggle/working/transcripts', 'zip', root_dir=OUTPUT_ROOT)
    size_mb = Path(archive).stat().st_size / 1024 / 1024
    print(f'Đã đóng gói: {archive} ({size_mb:.1f} MB)')
    print('Tải về ở panel Output bên phải, hoặc Save Version để giữ lại.')
else:
    print('Bỏ qua: kết quả đã nằm ở', OUTPUT_ROOT)